# Developer's App — Google Colab Edition
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/REPOSITORY/blob/main/Developers_App_Colab.ipynb)

Runs the **original, unmodified** Next.js 14 project on Colab and publishes it through a Cloudflare quick tunnel.

**How to use:** `Runtime → Run all`. The public URL is printed in the *Public access* cell (about 3–5 minutes on a fresh runtime).

All Colab-specific logic lives in these notebook cells. Project source files are cloned as-is and verified against `SHA256SUMS.txt` before anything runs.

## 1 · Project initialization

In [ ]:
# Configuration (the only place you might ever edit)
import os, pathlib
REPO        = "USERNAME/REPOSITORY"          # set automatically by publish.sh
BRANCH      = "main"
WORK        = pathlib.Path("/content/developers-app")
APP_DIR     = WORK / "app"
PORT        = 3000

USE_ZIP = REPO.startswith("USERNAME")   # no GitHub repo configured -> use the uploaded package zip instead
ZIP_NAME = "Developers_App_Colab_Package.zip"
print("Source:", ("uploaded zip " + ZIP_NAME) if USE_ZIP else REPO, "| Work dir:", WORK, "| Port:", PORT)

## 2 · Runtime and Node.js setup

In [ ]:
import subprocess, re, shutil

def sh(cmd, check=True, **kw):
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=True, **kw)

def node_ok():
    if not shutil.which("node"): return False
    v = sh("node -v", check=False).stdout.strip().lstrip("v").split(".")
    return (int(v[0]), int(v[1])) >= (18, 17)     # Next.js 14.2 requires >= 18.17

if not node_ok():
    print("Installing Node.js 20 (NodeSource)...")
    sh("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs")
assert node_ok(), "Node.js >= 18.17 is required"
print("node", sh("node -v").stdout.strip(), "| npm", sh("npm -v").stdout.strip())

# The app's /api/execute route spawns python3 for its code sandbox
print(sh("python3 --version").stdout.strip())

## 3 · Project file preparation (clone or uploaded zip, then integrity check)

In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
if USE_ZIP:
    zp = pathlib.Path("/content") / ZIP_NAME
    if not zp.exists():
        from google.colab import files
        print(f"Select {ZIP_NAME} to upload...")
        files.upload()                       # saved into /content
    if not zp.exists():
        raise SystemExit(f"{ZIP_NAME} not found in /content")
    WORK.mkdir(parents=True)
    sh(f"unzip -q {zp} -d {WORK}")
else:
    r = sh(f"git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git {WORK}", check=False)
    if r.returncode != 0:
        raise SystemExit("git clone failed:\n" + r.stderr)

# Verify every packaged source file against the checksums of the original project
r = sh(f"cd {APP_DIR} && sha256sum -c ../SHA256SUMS.txt --quiet", check=False)
if r.returncode != 0:
    raise SystemExit("INTEGRITY CHECK FAILED - files differ from the original:\n" + r.stdout + r.stderr)
n = sh(f"wc -l < {WORK}/SHA256SUMS.txt").stdout.strip()
print(f"Integrity OK: {n} files byte-identical to the original project")

## 4 · Dependency installation (exact versions from `package-lock.json`)

In [ ]:
r = sh(f"cd {APP_DIR} && npm ci --no-audit --no-fund", check=False)
print(r.stdout[-1500:])
if r.returncode != 0:
    raise SystemExit(r.stderr[-3000:])

## 5 · Environment and configuration
The project reads **no custom environment variables** and uses no external API keys (`supabase/` contains only an unused SQL schema), so no secrets are needed or exposed.

In [ ]:
os.environ["NEXT_TELEMETRY_DISABLED"] = "1"
os.environ["PORT"] = str(PORT)
print("No project-specific environment variables required.")

## 6 · Build

In [ ]:
r = sh(f"cd {APP_DIR} && npx next build", check=False)
print(r.stdout[-2500:])
if r.returncode != 0:
    raise SystemExit(r.stderr[-3000:])

## 7 · Application startup (background process)

In [ ]:
import time, socket, urllib.request

LOG = WORK / "next-server.log"
server = subprocess.Popen(
    f"cd {APP_DIR} && exec npx next start -p {PORT} -H 0.0.0.0",
    shell=True, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Server did not start. Log:\n" + LOG.read_text())
print(f"Next.js is up on http://127.0.0.1:{PORT}")

## 8 · Health checks and functionality verification

In [ ]:
import json, urllib.error

def req(path, data=None, base=f"http://127.0.0.1:{PORT}"):
    rq = urllib.request.Request(base + path, method="POST" if data is not None else "GET",
                                data=json.dumps(data).encode() if data is not None else None,
                                headers={"content-type": "application/json"})
    try:
        with urllib.request.urlopen(rq, timeout=15) as r: return r.status, r.read()
    except urllib.error.HTTPError as e: return e.code, e.read()

checks = []
def check(name, ok): checks.append(ok); print(("PASS" if ok else "FAIL"), name)

for p in ["/", "/account-selection", "/feed", "/login/developer", "/login/organization",
          "/signup/developer", "/signup/organization"]:
    check(f"page {p}", req(p)[0] == 200)
check("404 page", req("/does-not-exist")[0] == 404)
for p in ["/api/projects", "/api/certificates", "/api/opportunities", "/api/opportunities/apply"]:
    check(f"GET {p}", req(p)[0] == 200)
check("asset animation frame 1",   req("/animation/ezgif-frame-001.jpg")[0] == 200)
check("asset animation frame 300", req("/animation/ezgif-frame-300.jpg")[0] == 200)

s, b = req("/api/execute", {"language": "javascript", "code": "console.log(21*2)"})
check("sandbox JavaScript", json.loads(b).get("output") == "42")
s, b = req("/api/execute", {"language": "python", "code": "print(sum(range(10)))"})
check("sandbox Python", json.loads(b).get("output") == "45")
s, b = req("/api/execute", {"language": "javascript", "code": "require('child_process')"})
check("sandbox blocklist (403)", s == 403)
s, b = req("/api/opportunities/apply", {})
check("apply validation (400)", s == 400)
s, b = req("/api/opportunities/apply", {"opportunityId": "o", "developerUsername": "u"})
check("apply success", s == 200 and json.loads(b).get("success") is True)

print(f"\n{sum(checks)}/{len(checks)} checks passed")

## 9 · Public access (Cloudflare quick tunnel)

In [ ]:
CF = WORK / "cloudflared"
if not CF.exists():
    sh(f"curl -fsSL -o {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}")

TLOG = WORK / "tunnel.log"
tunnel = subprocess.Popen(f"exec {CF} tunnel --url http://127.0.0.1:{PORT} --no-autoupdate",
                          shell=True, stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TLOG.read_text())
    if m: PUBLIC_URL = m.group(0); break
    time.sleep(1)
if not PUBLIC_URL:
    raise SystemExit("Tunnel did not start. Log:\n" + TLOG.read_text()[-2000:])

from IPython.display import HTML, display
display(HTML(f'<h3>Public URL: <a href="{PUBLIC_URL}" target="_blank">{PUBLIC_URL}</a></h3>'
             '<p>This link is public to anyone who has it and lasts while this notebook keeps running.</p>'))

In [ ]:
# Verify through the PUBLIC tunnel (DNS for a new tunnel can take a few seconds)
for attempt in range(15):
    try:
        s, _ = req("/", base=PUBLIC_URL)
    except Exception:
        s = 0
    if s == 200: break
    time.sleep(2)
print("Public endpoint status:", s)
print("Public /api/projects  :", req("/api/projects", base=PUBLIC_URL)[0])

## 10 · Keep alive  
This cell runs until you stop it (■). Stopping it leaves the app running; use the next cell to shut everything down.

In [ ]:
print("App is live at:", PUBLIC_URL)
try:
    while True:
        time.sleep(60)
        if server.poll() is not None:
            print("Server exited. Log tail:\n" + LOG.read_text()[-1500:]); break
except KeyboardInterrupt:
    print("Stopped waiting. App is still running.")

## 11 · Troubleshooting and cleanup (manual)

In [ ]:
# Set to True and run this cell to stop the app and tunnel. Keep False for "Run all".
CLEANUP = False

print("--- server log (tail) ---");  print(LOG.read_text()[-1500:])
if "TLOG" in globals() and TLOG.exists():
    print("--- tunnel log (tail) ---");  print(TLOG.read_text()[-800:])
if CLEANUP:
    for name in ("tunnel", "server"):
        if name in globals(): globals()[name].terminate()
    sh("pkill -f 'next start' ; pkill -f cloudflared", check=False)
    print("Stopped.")